# Pilot — inisiasi anotasi otomatis kurma (palmseg) di Google Colab

Tujuan: menghasilkan **anotasi awal** (ellipse kurma) pada sampel kecil dari 20 epoch × 36 farm Pléiades, lalu Anda **mereviu manual di CVAT**. Bukan ground truth final. `run_002` = 36 farm × 20 epoch, inferensi paralel (6 proses, jendela w0 lalu w1).

Alur: `config → setup → manifest → pilih sampel → chip → inferensi (3 model × 2 GSD) → konsensus → paket review CVAT → skor review`.

- Runtime: **GPU (T4 cukup)**. Semua hasil penting disimpan ke Google Drive (`Thesis/pilot/<RUN_ID>`), jadi aman bila sesi putus. Jalankan ulang dari atas; sel yang sudah selesai akan dilewati.
- Model: repo `brakuta/Date-Palm-Tree-Segmentation-from-WorldView3-Satellite-Data` (kode Apache-2.0, bobot CC-BY-4.0 — cantumkan atribusi). Dilatih pada WorldView-3 ~0,3 m; data kita Pléiades 0,5 m → **ada domain gap**, karena itu diuji juga versi di-upsample ke 0,3 m.
- Model 8-band tidak dipakai (data kita 3/4 band). NIR tidak dipakai oleh model; hanya dipakai untuk memilih jendela sampel yang bervegetasi.

## 1. Konfigurasi

In [ ]:
import os, sys, json, re, subprocess, shutil, time, math
from pathlib import Path
import numpy as np, pandas as pd
if "google.colab" in sys.modules:          # mount dulu: mkdir di /content/drive sebelum mount membuat mount gagal/hang
    from google.colab import drive
    drive.mount("/content/drive")

RUN_ID   = "run_002"   # "run_003" = farm F014 saja, semua epoch, hanya w0 (jalankan run_stage(0), lewati sel 6b)
ROOT     = Path(os.environ.get("THESIS_ROOT", "/content/drive/MyDrive/Thesis"))
DATASETS = ROOT / "datasets"
WORK     = ROOT / "pilot" / RUN_ID          # persisten di Drive
WEIGHTS  = ROOT / "pilot" / "weights"       # cache bobot, dipakai lintas run
LOCAL    = Path(os.environ.get("LOCAL_DIR", "/content/work"))   # cepat, sementara
SEED     = 42

# --- sampel (run_002: 36 farm × 20 epoch; jendela diproses berurutan w0 lalu w1) ---
N_FARMS          = 36       # farm dipilih merata menurut luas
WINDOWS_PER_FARM = 2        # 1 paling bervegetasi + sisanya acak (lokasi SAMA di semua epoch)
CHIP_M           = 768      # sisi jendela (meter)
EPOCH_PICKS      = {"rgb": 14, "rgbn05": 5, "rgbn03": 1}   # jumlah epoch per grup band, tersebar waktu
BAND_ORDER      = {}       # override per service, mis. {"QATAR_2012_08_C050_PLE": "BGR"}; default: deteksi otomatis. R-G-B terverifikasi manual di 2012_08, 2022_01, 2025_04

# --- model & inferensi ---
N_PROCS       = 6          # 6 proses = 3,4x serial pada 1 A100, VRAM 9 GB (hasil benchmark). T4 (RAM 13 GB): pakai 1, 2 proses kena OOM di g030 (run_003)
BATCH_CHIPS   = 120        # chip per batch; resume per batch
MODELS         = ["segformer_b3_rgb", "upernet_swin_t_rgb", "upernet_vit_deit_s_rgb"]
PROB_THR       = 0.5
MIN_AREA_30CM  = 30         # px, nilai quickstart repo pada 0,3 m; diskalakan otomatis ke GSD lain
MIN_SEED_30CM  = 8          # px
REPO_URL       = "https://github.com/brakuta/Date-Palm-Tree-Segmentation-from-WorldView3-Satellite-Data.git"

# --- konsensus & review ---
MATCH_RADIUS_M   = 2.0      # pusat tajuk dianggap pohon yang sama bila < ini
PERSIST_RADIUS_M = 3.0      # toleransi registrasi antar epoch
REVIEW_TILE_M    = 100      # sisi tile review
N_REVIEW_TILES   = 40
REVIEW_UPSCALE   = 3        # perbesar tile agar nyaman dilihat di CVAT

ONLY_FARMS = None           # None = pilih N_FARMS merata (run_002)
if RUN_ID == "run_003":     # model/grid/tier/CHIP_M/MODELS sama dengan run_002 -> tier sebanding
    ONLY_FARMS = ["F014"]; WINDOWS_PER_FARM = 1; EPOCH_PICKS = {g: 99 for g in EPOCH_PICKS}   # 99 = semua epoch

for p in (WORK, WEIGHTS, LOCAL): p.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)
print("RUN_ID:", RUN_ID, "| DATASETS:", DATASETS, DATASETS.exists())

## 2. Setup: Drive, GPU, lingkungan

In [ ]:
def sh(cmd, cwd=None, check=True):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, executable="/bin/bash")
    for line in p.stdout: print(line, end="")
    p.wait()
    if check and p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return p.returncode

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
assert DATASETS.exists(), f"Folder dataset tidak ditemukan: {DATASETS}"
has_gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False) == 0
if not has_gpu: print("PERINGATAN: tidak ada GPU. Runtime > Change runtime type > GPU.")

try:
    import rasterio, geopandas, shapely, scipy, pyarrow, PIL
except ImportError:
    sh("pip -q install rasterio geopandas shapely scipy pyarrow pillow")
    import rasterio, geopandas, shapely, scipy, pyarrow, PIL

### 2b. Lingkungan palmseg (venv terisolasi)
Repo butuh `torch 2.1 + mmcv 2.1.0` (prebuilt, CUDA 12.1) dan `numpy<2`. Colab punya versi lain, jadi dibuat **venv Python 3.10 terpisah** dengan `uv`; kernel notebook tidak terganggu. Sekitar 5–8 menit pada sesi baru. Jika `mmcv` tidak menemukan wheel prebuilt, instalasi **gagal cepat** (tidak mengompilasi).

In [ ]:
VENV = Path("/content/palmenv"); PY = VENV / "bin/python"; PALM = VENV / "bin/palmseg"
REPO = Path("/content/palmseg_repo")
if not PALM.exists():
    sh("pip -q install uv")
    sh(f"uv venv {VENV} --python 3.10 --seed --clear")   # --clear: venv setengah jadi dari run gagal sebelumnya
    sh(f"uv pip install --python {PY} -q 'numpy<2' torch==2.1.0 torchvision==0.16.0")   # build PyPI Linux = CUDA 12.1
    sh(f"uv pip install --python {PY} -q --no-build mmengine 'mmcv==2.1.0' "
       f"-f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html")   # --no-build: gagal cepat, tidak kompilasi
    sh(f"uv pip install --python {PY} -q 'mmsegmentation>=1.2.0,<1.3'")
    if not REPO.exists(): sh(f"git clone --depth 1 {REPO_URL} {REPO}")
    sh(f"uv pip install --python {PY} -q -e {REPO}")
    # instal repo menarik numpy 2 + opencv baru; torch 2.1 butuh numpy<2 -> pin ulang
    sh(f"uv pip install --python {PY} -q 'numpy<2' 'opencv-python-headless<4.12' 'opencv-python<4.12'")
    sh(f"uv pip install --python {PY} -q ftfy regex")      # dibutuhkan selftest/model (tidak ada di requirements repo)
# mmseg 1.2.x sudah mendaftarkan LoadSingleRSImageFromFile (versi gdal); versi repo (rasterio) harus menimpa
sh(f"sed -i 's/@TRANSFORMS.register_module()/@TRANSFORMS.register_module(force=True)/' {REPO}/palmseg/transforms/loading.py")
# checkpoint upernet_vit memuat auxiliary_head sisa ADE20K (150 kelas) -> loader menolak; head ini hanya untuk training, abaikan (idempotent)
_lp = REPO / "palmseg/loader.py"; _s = _lp.read_text()
_old = "    model_sd = model.state_dict()\n    ignorable = ('num_batches_tracked',)\n"
if "auxiliary_head hanya dipakai" not in _s:
    assert _old in _s, "loader.py berubah; sesuaikan patch"
    _lp.write_text(_s.replace(_old, _old
        + "    # auxiliary_head hanya dipakai saat training; abaikan bila bentuknya beda\n"
        + "    state = {k: v for k, v in state.items() if not (k.startswith('auxiliary_head.') and k in model_sd\n"
        + "                                                  and tuple(v.shape) != tuple(model_sd[k].shape))}\n"
        + "    ignorable = ignorable + tuple(k for k in model_sd if k.startswith('auxiliary_head.') and k not in state)\n", 1))
sh(f"{PALM} doctor", cwd=REPO, check=False)
REPO_HEAD = subprocess.run(f"git -C {REPO} rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
print("palmseg commit:", REPO_HEAD)

### 2c. Unduh bobot (cache di Drive) dan uji cepat

In [ ]:
for m in MODELS:
    ck = WEIGHTS / f"{m}.safetensors"
    if not ck.exists(): sh(f"{PALM} download {m} --cache-dir {WEIGHTS}", cwd=REPO)
    sh(f"{PALM} inspect {ck}", cwd=REPO)          # harus: in_channels 3, num_classes 2, modality rgb
sh(f"{PALM} selftest --model {MODELS[0]} --checkpoint {WEIGHTS / (MODELS[0] + '.safetensors')} --device auto", cwd=REPO)

## 3. Manifest dari JSON sidecar
Membaca JSON saja (cepat di Drive). Mengabaikan sidecar `.azDownload-*`.

In [ ]:
rows, farm_wkt = [], {}
for jf in sorted(DATASETS.glob("QATAR_*/*/*.json") if not ONLY_FARMS else (j for f in ONLY_FARMS for j in DATASETS.glob(f"QATAR_*/{f}/*.json"))):
    if jf.name.startswith("."): continue
    d = json.loads(jf.read_text())
    m = re.match(r"QATAR_(\d{4})_(\d{2})_C(\d{3})_PLE", d["service_name"])
    tif = jf.with_suffix(".tif"); msk = jf.with_name(jf.stem + "_mask.tif")
    if not (tif.exists() and msk.exists()): continue
    res, nb = float(d["cell_size_x_raw"]), int(d["band_count"])
    grp = "rgb" if nb == 3 else ("rgbn03" if res < 0.4 else "rgbn05")
    e = d["extent"]
    rows.append(dict(service=d["service_name"], epoch=f"{m[1]}_{m[2]}", farm=str(d["farm_no"]), bands=nb, res=res,
                     grp=grp, cols=d["columns"], rws=d["rows"], tif=str(tif), msk=str(msk),
                     area_ha=d["columns"] * d["rows"] * res * res / 1e4))
    farm_wkt.setdefault(str(d["farm_no"]), d["farm_geometry_wkt"])
man = pd.DataFrame(rows)
man.to_parquet(WORK / "manifest.parquet")
print(len(man), "pasang  |", man.service.nunique(), "epoch |", man.farm.nunique(), "farm")
print(man.groupby("grp").agg(clips=("farm", "size"), epochs=("service", "nunique")))
exp = man.service.nunique() * man.farm.nunique()
if len(man) != exp: print(f"PERINGATAN: {len(man)} ditemukan, {exp} diharapkan — periksa file yang hilang")

## 4. Pilih sampel
- **Farm**: `N_FARMS` tersebar merata menurut luas.
- **Epoch**: tiap grup band, tersebar merata dalam waktu.
- **Jendela**: sisi `CHIP_M` (mengecil otomatis untuk farm kecil), dipilih sekali per farm pada epoch RGBN 0,5 m terbaru (NDVI untuk vegetasi), lalu **bounding box yang sama dipotong dari semua epoch** → dasar proxy persistensi dan kelak S4/S5.

In [ ]:
from rasterio.windows import Window, from_bounds
from rasterio.enums import Resampling
from affine import Affine
farm_med = man.groupby("farm").area_ha.median().sort_values()
fi = np.unique(np.linspace(0, len(farm_med) - 1, N_FARMS).round().astype(int))
FARMS = ONLY_FARMS or list(farm_med.index[fi])

pick = []
for g, k in EPOCH_PICKS.items():
    ep = sorted(man.loc[man.grp == g, "service"].unique())
    if not ep: continue
    idx = np.unique(np.linspace(0, len(ep) - 1, min(k, len(ep))).round().astype(int))
    pick += [ep[i] for i in idx]
EPOCHS = sorted(pick)
print("farm:", FARMS); print("epoch:", EPOCHS)

ref_service = sorted(man.loc[man.grp == "rgbn05", "service"].unique())[-1]
print("epoch referensi (NDVI):", ref_service)

def choose_windows(farm, n=WINDOWS_PER_FARM, dec_m=8.0):
    r = man[(man.farm == farm) & (man.service == ref_service)].iloc[0]
    with rasterio.open(r.tif) as s, rasterio.open(r.msk) as sm:
        f = max(1, int(round(dec_m / s.res[0])))
        h, w = s.height // f, s.width // f
        a = s.read(out_shape=(s.count, h, w), resampling=Resampling.average).astype("float32")
        mk = sm.read(1, out_shape=(h, w), resampling=Resampling.nearest) > 0
        T = s.transform * Affine.scale(s.width / w, s.height / h)
    red = a[0] if a[0][mk].mean() > a[2][mk].mean() else a[2]      # band merah: lihat catatan urutan band di sel chip
    nir = a[3]
    veg = (((nir - red) / (nir + red + 1e-6)) > 0.2) & mk
    c = int(min(round(CHIP_M / dec_m), 0.9 * min(h, w)))      # farm kecil → jendela lebih kecil
    if c < 8: return []
    cs = lambda x: np.pad(x.astype("float64").cumsum(0).cumsum(1), ((1, 0), (1, 0)))
    I_m, I_v = cs(mk), cs(veg)
    box = lambda I, y, x: I[y + c, x + c] - I[y, x + c] - I[y + c, x] + I[y, x]
    cand = [(y, x, box(I_m, y, x) / c**2, box(I_v, y, x) / c**2)
            for y in range(0, h - c + 1, max(1, c // 2)) for x in range(0, w - c + 1, max(1, c // 2))]
    cand = [t for t in cand if t[2] >= 0.9] or [t for t in cand if t[2] >= 0.6]   # farm tak beraturan: longgarkan
    if not cand: return []
    out = [max(cand, key=lambda t: t[3])]
    rest = [t for t in cand if abs(t[0] - out[0][0]) >= c or abs(t[1] - out[0][1]) >= c]
    while len(out) < n and rest:
        t = rest.pop(int(rng.integers(len(rest)))); out.append(t)
        rest = [q for q in rest if abs(q[0] - t[0]) >= c or abs(q[1] - t[1]) >= c]
    res = []
    for y, x, cov, vg in out:
        x0, y1 = T * (x, y); x1, y0 = T * (x + c, y + c)
        res.append((min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)))   # xmin,ymin,xmax,ymax (m)
    return res

WIN = {fm: choose_windows(fm) for fm in FARMS}
print({k: len(v) for k, v in WIN.items()})
(WORK / "windows.json").write_text(json.dumps(WIN))

## 5. Buat chip
- RGB 8-bit GeoTIFF; NIR dibuang. **Urutan band dideteksi otomatis** (gurun: pasir lebih merah daripada biru → band-1 > band-3 = RGB). Pengecekan awal pada 12 sampel (3-band dan 4-band) konsisten dengan **R,G,B,NIR**, bukan B,G,R,NIR seperti tertulis di spec; metadata `band_order` tidak dipakai. Hasil deteksi dicetak per epoch dan bisa di-override lewat `BAND_ORDER`; cek visual di sel pratinjau (pasir harus tampak cokelat/kuning, bukan biru).
- Tanpa contrast stretch (kontrak input model).
- `g050/` = chip 0,5 m asli; `g030/` = chip asli 0,3 m + versi **di-upsample ke 0,3 m** (akhiran `__up`).

In [ ]:
CH = LOCAL / "chips"
for g in ("g050", "g030"): (CH / g).mkdir(parents=True, exist_ok=True)
chip_rows = []

def to_rgb(arr, order):          # band ke-4 (NIR) diabaikan
    return arr[[2, 1, 0]] if order == "BGR" else arr[:3]

for svc in EPOCHS:
    sub = man[(man.service == svc) & (man.farm.isin(FARMS))]
    loaded = []
    for r in sub.itertuples():
        for k, (x0, y0, x1, y1) in enumerate(WIN[r.farm]):
            with rasterio.open(r.tif) as s:
                win = from_bounds(x0, y0, x1, y1, s.transform).round_offsets().round_lengths()
                arr = s.read(window=win, boundless=True, fill_value=0)
                T, crs = s.window_transform(win), s.crs
            valid = float((arr.max(0) > 0).mean())
            if valid < 0.5: print(f"lewati {r.farm}/{r.epoch}/w{k}: footprint {valid:.0%}"); continue
            loaded.append((r, k, arr, T, crs, valid))
    if not loaded: continue
    # urutan band TIDAK dipercaya dari metadata: di gurun pasir lebih merah daripada biru → band-1 > band-3 berarti RGB
    votes = [np.mean(a[0][a[0] > 0]) > np.mean(a[2][a[2] > 0]) for _, _, a, _, _, _ in loaded if (a[0] > 0).any()]
    order = BAND_ORDER.get(svc, "RGB" if np.mean(votes) >= 0.5 else "BGR")
    print(f"{svc}: {loaded[0][0].bands}-band → {order}  (suara RGB {np.mean(votes):.0%} dari {len(votes)} chip)")
    for r, k, arr, T, crs, valid in loaded:
        rgb = to_rgb(arr, order)
        cid = f"{r.farm}__{r.epoch}__w{k}"
        prof = dict(driver="GTiff", count=3, dtype="uint8", height=rgb.shape[1], width=rgb.shape[2],
                    crs=crs, transform=T, compress="deflate")
        g = "g030" if r.res < 0.4 else "g050"
        with rasterio.open(CH / g / f"{cid}.tif", "w", **prof) as d: d.write(rgb)
        chip_rows.append(dict(chip=cid, farm=r.farm, epoch=r.epoch, service=svc, grp=r.grp, win=k, native_res=r.res,
                              variant="native", g=g, x0=T.c, y1=T.f, h=rgb.shape[1], w=rgb.shape[2], valid=valid,
                              order=order))
        if r.res > 0.4:   # versi di-upsample ke 0,3 m
            f = r.res / 0.30
            oh, ow = round(rgb.shape[1] * f), round(rgb.shape[2] * f)
            with rasterio.open(CH / "g050" / f"{cid}.tif") as s2:
                up = s2.read(out_shape=(3, oh, ow), resampling=Resampling.cubic).clip(0, 255).astype("uint8")
            prof.update(height=oh, width=ow, transform=T * Affine.scale(1 / f))
            with rasterio.open(CH / "g030" / f"{cid}__up.tif", "w", **prof) as d: d.write(up)
            chip_rows.append({**chip_rows[-1], "variant": "up", "g": "g030", "h": oh, "w": ow})
chips = pd.DataFrame(chip_rows)
chips.to_parquet(WORK / "chips.parquet")
print(len(chips), "chip |", chips.groupby(["g", "variant"]).size().to_dict())

In [ ]:
# pratinjau + statistik band (cek radiometri kasar dan urutan band)
import matplotlib.pyplot as plt
sel = chips[chips.variant == "native"].groupby("grp").head(1)
fig, ax = plt.subplots(1, len(sel), figsize=(5 * len(sel), 5), squeeze=False)
for a, r in zip(ax[0], sel.itertuples()):
    with rasterio.open(CH / r.g / f"{r.chip}.tif") as s:
        im = s.read(out_shape=(3, 600, 600)); mean = [float(im[i][im[i] > 0].mean()) for i in range(3)]
    a.imshow(np.moveaxis(im, 0, -1)); a.set_title(f"{r.chip} [{r.grp}]\nmean RGB={np.round(mean)}", fontsize=8); a.axis("off")
plt.show()

## 6. Inferensi: 3 model × (GSD asli, upsample 0,3 m)
Paralel: `N_PROCS` proses per batch `BATCH_CHIPS` chip pada satu GPU; resume per batch (chip yang `*_trees.gpkg`-nya sudah ada di Drive dilewati). Jendela diproses berurutan: **w0 dulu** (sel ini), evaluasi, lalu **w1** (sel di bawahnya).

In [ ]:
# Inferensi paralel bertahap: jendela w0 dulu, lalu w1. Resume per batch (cek gpkg di Drive).
import glob, os, time, subprocess, shutil
OUT = LOCAL / "out"; RAW = WORK / "detections_raw"; RAW.mkdir(exist_ok=True)
GSD = {"g050": 0.5, "g030": 0.3}
run_log = []

def chip_stem(r): return r.chip + ("__up" if r.variant == "up" else "")

def pending(m, g, win):
    done = {p.name[: -len("_trees.gpkg")] for p in (RAW / m / g).glob("*_trees.gpkg")} if (RAW / m / g).exists() else set()
    return [s for s in (chip_stem(r) for r in chips[(chips.g == g) & (chips.win == win)].itertuples()) if s not in done]

def run_batch(m, g, stems, tag):
    gsd = GSD[g]; mina = max(4, round(MIN_AREA_30CM * (0.3 / gsd) ** 2)); sd = max(2, round(MIN_SEED_30CM * 0.3 / gsd))
    root = OUT / m / g / tag; shutil.rmtree(root, ignore_errors=True); root.mkdir(parents=True)
    procs = []
    for k in range(N_PROCS):
        part = stems[k::N_PROCS]
        if not part: continue
        ind = root / f"in_{k}"; ind.mkdir()
        for s in part: os.symlink(CH / g / f"{s}.tif", ind / f"{s}.tif")
        cmd = (f"{PALM} segment-and-extract --model {m} --checkpoint {WEIGHTS / (m + '.safetensors')} --input-dir {ind} --pattern '*.tif' "
               f"--out {root}/out_{k} --summary-csv {root}/s_{k}.csv --min-area {mina} --min-seed-distance {sd} --prob-thr {PROB_THR} "
               f"--tile 512 --overlap 128 --device auto")
        procs.append(subprocess.Popen(cmd, shell=True, cwd=REPO, stdout=open(root / f"log_{k}.txt", "w"), stderr=subprocess.STDOUT, executable="/bin/bash"))
    for p in procs: p.wait()
    bad = [p.returncode for p in procs if p.returncode]
    (RAW / m / g).mkdir(parents=True, exist_ok=True)
    got = list(root.rglob("*_trees.gpkg"))
    for f in got: shutil.copy(f, RAW / m / g / f.name)
    for f in root.glob("s_*.csv"): shutil.copy(f, RAW / f"summary_{m}_{g}_{tag}_{f.stem}.csv")
    shutil.rmtree(root, ignore_errors=True)       # label/prob raster besar; tidak dipertahankan
    return len(got), bad

def run_stage(win, models=MODELS, grids=("g050", "g030")):
    for g in grids:
        for m in models:
            todo = pending(m, g, win)
            print(f"[w{win}] {m}/{g}: {len(todo)} chip belum selesai", flush=True)
            for i in range(0, len(todo), BATCH_CHIPS):
                t = time.time(); b = todo[i:i + BATCH_CHIPS]
                n, bad = run_batch(m, g, b, f"w{win}_{i // BATCH_CHIPS}")
                row = dict(win=win, model=m, g=g, batch=i // BATCH_CHIPS, chips=len(b), gpkg=n, rc_bad=str(bad), sec=round(time.time() - t, 1))
                run_log.append(row); print(row, flush=True)
                pd.DataFrame(run_log).to_csv(WORK / "runs_this_session.csv", index=False)
                if n != len(b): raise RuntimeError(f"batch tidak lengkap: {n}/{len(b)} gpkg; cek log (rc={bad})")
run_stage(0)
print("STAGE w0 SELESAI")
runs = run_log   # dipakai sel metadata run

### 6b. Jendela w1 (jalankan setelah w0 dievaluasi)
Lewati sel ini bila hanya w0 yang dipakai; tidak ada efek bila tidak dijalankan. Resume otomatis.

In [ ]:
run_stage(1)
print("STAGE w1 SELESAI")

## 7. Kumpulkan deteksi, saring dengan batas farm

In [ ]:
import geopandas as gpd, shapely
from shapely import wkt as shp_wkt
FARM_POLY = {k: shp_wkt.loads(v) for k, v in farm_wkt.items()}
recs = []
for f in sorted(RAW.rglob("*_trees.gpkg")):
    m, g = f.parent.parent.name, f.parent.name
    stem = f.name[: -len("_trees.gpkg")]
    variant = "up" if stem.endswith("__up") else "native"
    cid = stem[:-4] if variant == "up" else stem
    gdf = gpd.read_file(f)
    if len(gdf) == 0: continue
    if gdf.crs is not None and gdf.crs.to_epsg() != 2932: gdf = gdf.to_crs(2932)
    c = gdf.geometry.centroid
    farm = cid.split("__")[0]
    inside = shapely.contains_xy(FARM_POLY[farm], c.x.values, c.y.values)
    d = pd.DataFrame(dict(chip=cid, model=m, variant=variant, x=c.x.values, y=c.y.values,
                          score=gdf["score"].values if "score" in gdf else np.nan,
                          circ=gdf["circularity"].values if "circularity" in gdf else np.nan,
                          diam_m=gdf["diam_m"].values if "diam_m" in gdf else np.nan))[inside]
    recs.append(d)
det = pd.concat(recs, ignore_index=True)
det["cfg"] = det.model + "|" + det.variant
det.to_parquet(WORK / "detections.parquet")
print(det.groupby(["model", "variant"]).size().unstack())
print("kolom gpkg:", list(gdf.columns))

## 8. Konsensus antar model/skala + proxy tanpa ground truth
Titik dari berbagai konfigurasi yang jatuh dalam `MATCH_RADIUS_M` digabung. `support` = jumlah konfigurasi yang setuju (dari semua konfigurasi yang berjalan pada chip itu).

In [ ]:
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

def consensus(d, ncfg):
    P = d[["x", "y"]].to_numpy(); n = len(P)
    pairs = cKDTree(P).query_pairs(MATCH_RADIUS_M, output_type="ndarray")
    A = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(n, n)) if len(pairs) else coo_matrix((n, n))
    _, lab = connected_components(A, directed=False)
    d = d.assign(comp=lab)
    g = d.groupby("comp").agg(x=("x", "mean"), y=("y", "mean"), score=("score", "mean"),
                              diam_m=("diam_m", "mean"), support=("cfg", "nunique"))
    g["frac"] = g.support / ncfg
    g["tier"] = np.where(g.frac >= 2 / 3, "high", np.where(g.frac >= 1 / 3, "med", "low"))
    return g.reset_index(drop=True)

cons = []
for cid, d in det.groupby("chip"):
    ncfg = len(MODELS) * int((chips.chip == cid).sum())    # konfigurasi yang berjalan pada chip ini
    c = consensus(d, ncfg); c.insert(0, "chip", cid); cons.append(c)
cons = pd.concat(cons, ignore_index=True)
cons = cons.merge(chips[chips.variant == "native"][["chip", "farm", "epoch", "service", "grp", "win"]], on="chip")
cons.to_parquet(WORK / "consensus.parquet")
print(cons.groupby(["grp", "tier"]).size().unstack(fill_value=0))

In [ ]:
# (a) kesepakatan antar konfigurasi (F1 pada pencocokan titik) — rata-rata per pasangan
def f1(A, B, r=MATCH_RADIUS_M):
    if len(A) == 0 or len(B) == 0: return np.nan
    ta, tb = cKDTree(A), cKDTree(B)
    pa = (tb.query(A, distance_upper_bound=r)[0] < r).mean(); pb = (ta.query(B, distance_upper_bound=r)[0] < r).mean()
    return 2 * pa * pb / (pa + pb) if pa + pb else 0.0
cfgs = sorted(det.cfg.unique()); M = pd.DataFrame(index=cfgs, columns=cfgs, dtype=float)
for a in cfgs:
    for b in cfgs:
        v = [f1(da[["x", "y"]].to_numpy(), db[["x", "y"]].to_numpy())
             for cid, da in det[det.cfg == a].groupby("chip")
             for db in [det[(det.cfg == b) & (det.chip == cid)]]]
        M.loc[a, b] = np.nanmean(v) if len(v) else np.nan
print("Kesepakatan F1 antar konfigurasi:"); print(M.round(2).to_string())
M.to_csv(WORK / "agreement_f1.csv")

# (b) hitungan per farm-jendela-epoch (konsensus high+med)
good = cons[cons.tier.isin(["high", "med"])]
cnt = good.groupby(["farm", "win", "epoch"]).size().unstack("epoch", fill_value=0)
print("Hitungan pohon (high+med):"); print(cnt)
cnt.to_csv(WORK / "counts_by_epoch.csv")

# (c) persistensi: porsi titik di epoch referensi (terbaru) yang muncul lagi di epoch lain (<= PERSIST_RADIUS_M)
pers = []
for (farm, win), d in good.groupby(["farm", "win"]):
    eps = sorted(d.epoch.unique()); ref = d[d.epoch == eps[-1]][["x", "y"]].to_numpy()
    for e in eps:
        o = d[d.epoch == e][["x", "y"]].to_numpy()
        if len(ref) and len(o):
            pers.append(dict(farm=farm, win=win, epoch=e, ref=eps[-1], n_ref=len(ref), n=len(o),
                             recall_vs_ref=(cKDTree(o).query(ref, distance_upper_bound=PERSIST_RADIUS_M)[0] < PERSIST_RADIUS_M).mean(),
                             prec_vs_ref=(cKDTree(ref).query(o, distance_upper_bound=PERSIST_RADIUS_M)[0] < PERSIST_RADIUS_M).mean()))
pers = pd.DataFrame(pers); pers.to_csv(WORK / "persistence.csv", index=False)
print("Persistensi vs epoch referensi (rendah = deteksi tidak stabil ATAU perubahan nyata ATAU registrasi buruk):")
print(pers.pivot_table(index=["farm", "win"], columns="epoch", values="recall_vs_ref").round(2))

## 9. Paket review untuk CVAT

Memilih `N_REVIEW_TILES` tile (`REVIEW_TILE_M` m) dengan cukup banyak pohon, tersebar di grup band. Hasil: gambar JPG (diperbesar), `annotations.xml` (format *CVAT for images 1.1*, ellipse), dan `labels.json`.

**Di CVAT:** buat task baru → Constructor → tab *Raw* → tempel `labels.json` → unggah folder `images/` → *Actions → Upload annotations → CVAT 1.1* → pilih `annotations.xml`.
Label: `palm_high` / `palm_med` / `palm_low` (tingkat kesepakatan model), `palm_missed` (pohon yang terlewat, Anda gambar). Hapus yang salah, geser/ubah ukuran yang kurang pas.

⚠ Format impor CVAT ini belum diuji dari sisi saya; bila gagal impor, `review_points.csv` berisi semua titik untuk dipakai alternatif.

In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image
REV = WORK / "cvat_pack"; (REV / "images").mkdir(parents=True, exist_ok=True)
nat = chips[chips.variant == "native"].set_index("chip")

cand = []
for cid, d in cons.groupby("chip"):
    r = nat.loc[cid]; res = r.native_res
    nx = int(REVIEW_TILE_M / res)
    for ty in range(0, r.h - nx + 1, nx):
        for tx in range(0, r.w - nx + 1, nx):
            x0, y1 = r.x0 + tx * res, r.y1 - ty * res; x1, y0 = x0 + nx * res, y1 - nx * res
            q = d[(d.x >= x0) & (d.x < x1) & (d.y > y0) & (d.y <= y1)]
            if len(q) >= 10: cand.append(dict(chip=cid, tx=tx, ty=ty, nx=nx, n=len(q), grp=r.grp, res=res))
cand = pd.DataFrame(cand)
assert len(cand), "tidak ada tile kandidat — periksa deteksi"
# bulatkan merata antar grup band
cand["rk"] = cand.groupby("grp").cumcount()
sel = cand.sample(frac=1, random_state=SEED).assign(rk=lambda x: x.groupby("grp").cumcount()).sort_values(["rk", "grp"]).head(N_REVIEW_TILES).reset_index(drop=True)
sel["tile"] = [f"t{i:03d}_{c}_{a}_{b}" for i, (c, a, b) in enumerate(zip(sel.chip, sel.tx, sel.ty))]

root = ET.Element("annotations"); ET.SubElement(root, "version").text = "1.1"
pts_out = []
COLORS = {"palm_high": "#2ecc71", "palm_med": "#f1c40f", "palm_low": "#e74c3c", "palm_missed": "#3498db"}
for i, t in enumerate(sel.itertuples()):
    r = nat.loc[t.chip]; res = t.res; U = REVIEW_UPSCALE
    with rasterio.open(CH / r.g / f"{t.chip}.tif") as s:
        im = s.read(window=Window(t.tx, t.ty, t.nx, t.nx))
    v = im[im > 0]; lo, hi = np.percentile(v, (2, 98)) if v.size else (0, 255)        # stretch HANYA untuk tampilan
    disp = np.clip((im.astype("float32") - lo) / max(hi - lo, 1) * 255, 0, 255).astype("uint8")
    Image.fromarray(np.moveaxis(disp, 0, -1)).resize((t.nx * U, t.nx * U), Image.BICUBIC).save(REV / "images" / f"{t.tile}.jpg", quality=92)
    el = ET.SubElement(root, "image", id=str(i), name=f"{t.tile}.jpg", width=str(t.nx * U), height=str(t.nx * U))
    x0, y1 = r.x0 + t.tx * res, r.y1 - t.ty * res
    q = cons[(cons.chip == t.chip) & (cons.x >= x0) & (cons.x < x0 + t.nx * res) & (cons.y > y1 - t.nx * res) & (cons.y <= y1)]
    for p in q.itertuples():
        cx, cy = (p.x - x0) / res * U, (y1 - p.y) / res * U
        rad = max(3.0, (p.diam_m if p.diam_m == p.diam_m else 6.0) / 2 / res * U)
        ET.SubElement(el, "ellipse", label=f"palm_{p.tier}", source="auto", occluded="0",
                      cx=f"{cx:.2f}", cy=f"{cy:.2f}", rx=f"{rad:.2f}", ry=f"{rad:.2f}", rotation="0", z_order="0")
        pts_out.append(dict(tile=t.tile, chip=t.chip, x=p.x, y=p.y, tier=p.tier, support=p.support, score=p.score, diam_m=p.diam_m))
ET.ElementTree(root).write(REV / "annotations.xml", encoding="utf-8", xml_declaration=True)
(REV / "labels.json").write_text(json.dumps([{"name": k, "color": c, "type": "ellipse", "attributes": []} for k, c in COLORS.items()], indent=2))
pd.DataFrame(pts_out).to_csv(REV / "review_points.csv", index=False)
sel["x0"] = [nat.loc[c].x0 + tx * rs for c, tx, rs in zip(sel.chip, sel.tx, sel.res)]
sel["y1"] = [nat.loc[c].y1 - ty * rs for c, ty, rs in zip(sel.chip, sel.ty, sel.res)]
sel.to_parquet(REV / "tiles.parquet")
print(len(sel), "tile,", len(pts_out), "ellipse →", REV)
print(sel.groupby("grp").size().to_dict())

## 10. (Setelah review di CVAT) Skor hasil review
Export dari CVAT: *Export task dataset → CVAT for images 1.1*, simpan XML ke `cvat_pack/reviewed.xml`. Sel ini menghitung **precision per tingkat kesepakatan** dan **perkiraan recall** pada tile yang direview, dan menyimpan ellipse hasil review sebagai ground truth awal (`reviewed_gt.parquet`).

Catatan: ground truth ini berasal dari tile yang dipilih (≥10 pohon), jadi tidak mewakili seluruh farm; dan **jangan memakai konsensus model sebagai test set** untuk evaluasi A/PGS — gunakan hasil review ini.

In [ ]:
import xml.etree.ElementTree as ET
from scipy.spatial import cKDTree
REV = WORK / "cvat_pack"   # self-contained: sel ini bisa dijalankan setelah c02+c04 saja

def score_review(xml_paths):
    tiles = pd.read_parquet(REV / "tiles.parquet").set_index("tile")
    auto = pd.read_csv(REV / "review_points.csv")
    gt, seen = [], set()
    for im in (i for xp in xml_paths for i in ET.parse(xp).getroot().iter("image")):   # gabung semua reviewed*.xml
        t = im.get("name")[:-4]
        if t not in tiles.index: continue
        seen.add(t)   # hanya tile yang diunggah/direview dihitung
        r, U = tiles.loc[t], REVIEW_UPSCALE
        for sh_ in im:
            if sh_.tag == "ellipse": cx, cy = float(sh_.get("cx")), float(sh_.get("cy"))
            elif sh_.tag == "box": cx, cy = (float(sh_.get("xtl")) + float(sh_.get("xbr"))) / 2, (float(sh_.get("ytl")) + float(sh_.get("ybr"))) / 2
            elif sh_.tag in ("polygon", "polyline", "points"):
                p = np.array([[float(v) for v in a.split(",")] for a in sh_.get("points").split(";")]); cx, cy = p.mean(0)
            else: continue
            gt.append(dict(tile=t, label=sh_.get("label"), x=r.x0 + cx / U * r.res, y=r.y1 - cy / U * r.res))
    gt = pd.DataFrame(gt, columns=["tile", "label", "x", "y"])
    auto = auto[auto.tile.isin(seen)]
    kept = []
    for t, a in auto.groupby("tile"):
        g = gt[(gt.tile == t) & (gt.label != "palm_missed")]
        d = cKDTree(g[["x", "y"]].to_numpy()).query(a[["x", "y"]].to_numpy(), distance_upper_bound=MATCH_RADIUS_M)[0] if len(g) else np.full(len(a), np.inf)
        kept.append(a.assign(kept=d < MATCH_RADIUS_M))
    auto = pd.concat(kept)
    prec = auto.groupby("tier").kept.agg(["mean", "size"]).rename(columns={"mean": "precision", "size": "n_auto"})
    n_kept = int(auto.kept.sum()); n_gt = len(gt)
    print(f"Tile direview: {len(seen)} dari {len(tiles)}")
    print("Precision per tingkat:"); print(prec.round(3))
    print("Precision per tile x tingkat (n kecil \u2192 jangan overinterpretasi):")
    print(auto.groupby(["tile", "tier"]).kept.mean().unstack().round(2))
    print(f"Precision keseluruhan {auto.kept.mean():.3f} | ellipse akhir {n_gt} | dipertahankan dari auto {n_kept} | "
          f"ditambah manual {n_gt - n_kept} \u2192 recall auto \u2248 {n_kept / max(n_gt, 1):.3f}")
    gt.to_parquet(WORK / "reviewed_gt.parquet"); prec.to_csv(WORK / "review_precision_by_tier.csv")
    return auto, gt

_rv = sorted(REV.glob("reviewed*.xml"))   # tiap file = satu batch tile; tile tidak boleh tumpang tindih antar file
if _rv: print("XML:", [f.name for f in _rv]); auto_r, gt_r = score_review(_rv)
else: print("Belum ada reviewed*.xml \u2014 lakukan review di CVAT dulu, lalu jalankan ulang sel ini.")

In [ ]:
# Recall seed per tier, hanya pada tile "lengkap" (ada palm_missed = Anda menandai semua pohon terlewat di tile itu).
# GT = ellips akhir hasil review (dipertahankan + palm_missed); recall tier S = porsi GT yang ada titik auto tier S dalam MATCH_RADIUS_M.
_full = {t for t in gt_r.tile.unique() if t[:4] in ("t017","t019","t024")}   # tile batch recall: semua pohon terlewat ditandai oleh reviewer
rows = []
for name, S in {"high": ["high"], "high+med": ["high", "med"], "semua": ["high", "med", "low"]}.items():
    hit = n = 0
    for t in _full:
        g = gt_r[gt_r.tile == t]; a = auto_r[(auto_r.tile == t) & auto_r.tier.isin(S)]
        d = cKDTree(a[["x", "y"]].to_numpy()).query(g[["x", "y"]].to_numpy(), distance_upper_bound=MATCH_RADIUS_M)[0] if len(a) else np.full(len(g), np.inf)
        hit += int((d < MATCH_RADIUS_M).sum()); n += len(g)
    rows.append(dict(seed=name, recall=round(hit / max(n, 1), 3), gt_pohon=n, tile_lengkap=len(_full)))
print(pd.DataFrame(rows).to_string(index=False))

## 11. Catat metadata run

In [ ]:
import platform
def pv(pkg): return subprocess.run(f"{PY} -c 'import {pkg};print({pkg}.__version__)'", shell=True, capture_output=True, text=True).stdout.strip()
meta = dict(run_id=RUN_ID, time=time.strftime("%Y-%m-%d %H:%M:%S"), python=platform.python_version(),
            gpu=subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip(),
            torch=pv("torch"), mmcv=pv("mmcv"), mmseg=pv("mmseg"), palmseg_commit=REPO_HEAD,
            farms=FARMS, epochs=EPOCHS, models=MODELS, chip_m=CHIP_M, seed=SEED,
            params=dict(prob_thr=PROB_THR, min_area_30cm=MIN_AREA_30CM, min_seed_30cm=MIN_SEED_30CM, match_r=MATCH_RADIUS_M),
            n_chips=len(chips), n_detections=len(det), runs=runs)
(WORK / "run_meta.json").write_text(json.dumps(meta, indent=2, default=str))
print(json.dumps(meta, indent=2, default=str)[:1500])